# Contrato e integración de RB-1521 en HUC

**Dataset:** RB-1521 · **Cuaderno:** 05 · **Tema:** qué puede publicar el expediente y qué no hereda de BES-jun

## Introducción

BES-jun publica PIP, THD y un caudal de intersección, y enciende el semáforo con reglas de caudal escrito. RB-1521 no tiene esas columnas ni esa curva. Integrarlo copiando la pantalla de BES-jun mostraría presiones que el libro no calcula. El contrato de este expediente publica el reparto de caudal y deja el semáforo sin una clase operacional.

## Objetivos

Fijar el identificador, las salidas y la condición de la lámpara, y comprobar que un pozo de BES-jun no puede entrar a este motor.

## Metodología

La solicitud lleva `dataset_id` y `pozo_id`. El identificador de la serie es `RB-1521` porque el archivo no trae otro. El adaptador de HUC carga `src/evaluacion/motor.py` por ruta de archivo, con un nombre de módulo distinto del motor de BES-jun, para que los dos expedientes no compartan el import `evaluacion.motor`.

## Salidas

El modelo `M-RB1521-PARTICION` devuelve Q-fluidos, BSW, Q-water, Q-oil, el número de días de la serie y el número de días en que la partición cierra. No devuelve PIP ni THD. `recomendacion_operacional_autorizada` es falso. La clasificación de la consola queda en `REGISTRO`: hay un resultado y no hay una regla de caudal que lo pinte de verde, ámbar o rojo.


In [1]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from evaluacion.motor import evaluar_solicitud
salida = evaluar_solicitud({"dataset_id": "RB-1521", "pozo_id": "RB-1521"})
valores = salida["modelos"][0]["valores"]
print(sorted(valores))
print("autorizada", salida["recomendacion_operacional_autorizada"])
print("sin regla", salida["resultados_sin_regla"])
print("pozo ajeno", evaluar_solicitud({"pozo_id": "QF-084"})["estado_evaluacion"])


['bsw', 'dias_identidad_exacta', 'dias_serie', 'q_fluidos', 'q_oil', 'q_water']
autorizada False
sin regla True
pozo ajeno entrada_invalida


## Resultados

Las claves publicadas son las seis de la partición y del conteo. No aparece `pip_ft`. Un identificador QF-084 termina en `entrada_invalida`. La serie y el pozo de BES-jun siguen en su propio motor.

## Restricciones que quedan abiertas

No hay curva H-Q de la S11000N en el libro, así que no hay ventana de operación que trasladar al semáforo. IP y P-SUT no reproducen las expresiones ensayadas. P-14, la pregunta sobre el lugar de BES-jun frente a este D1 en el tablero de asesor-esp, sigue sin respuesta y no se cierra en este cuaderno. El expediente no pasa a `procesadas`.

## Conclusión

RB-1521 queda integrable como serie de un conjunto de equipos, con un cálculo de partición verificable y sin un modelo de bomba. El visualizador debe mostrar esas salidas y no las de BES-jun.
